In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

%matplotlib inline

In [11]:
# 1. Đọc dữ liệu
df = pd.read_csv("Data_CarPrice.csv")

# 2. Làm sạch dữ liệu cơ bản (Thay thế hoàn toàn file preprocess.py cũ)
def clean_data_final(data):
    temp = data.copy()
    # Chuyển đổi AskPrice và kmDriven về dạng số
    for col in ['AskPrice', 'kmDriven']:
        if col in temp.columns:
            temp[col] = pd.to_numeric(temp[col].astype(str).str.replace(r'[^\d]', '', regex=True), errors='coerce')
    
    # Loại bỏ hàng thiếu giá, điền thiếu kmDriven bằng median
    temp = temp.dropna(subset=['AskPrice'])
    temp['kmDriven'] = temp['kmDriven'].fillna(temp['kmDriven'].median())
    temp['Age'] = temp['Age'].clip(lower=0)
    return temp

df_clean = clean_data_final(df)

# 3. Chia Features và Target
# CHÚ Ý: Dùng np.log1p cho target để đạt R2 > 85%
features_num = ['Age', 'kmDriven', 'km_per_year', 'post_year', 'post_month']
features_cat = ['Transmission', 'Owner', 'FuelType', 'Brand', 'model']

X = df_clean[features_num + features_cat]
y = np.log1p(df_clean['AskPrice']) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# 4. Tạo bộ xử lý cột (Scaling cho số, OneHot cho chữ)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ])

# 5. Tạo Pipeline kết hợp (Đúng yêu cầu thuần Sklearn)
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ElasticNet(max_iter=5000, random_state=42))
])

# 6. Grid Search để tìm bộ tham số tối ưu nhất
param_grid = {
    'regressor__alpha': [0.001, 0.01, 0.1],
    'regressor__l1_ratio': [0.1, 0.5, 0.9]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Tham số tốt nhất tìm được: {grid_search.best_params_}")

Tham số tốt nhất tìm được: {'regressor__alpha': 0.001, 'regressor__l1_ratio': 0.1}


In [9]:
# 7. Dự đoán và đánh giá trên tập Test
y_pred_log = grid_search.predict(X_test)
r2_final = r2_score(y_test, y_pred_log)

# Chuyển ngược từ log về giá thực tế để tính sai số VNĐ
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)
mae = mean_absolute_error(y_test_real, y_pred_real)

print(f"--- BÁO CÁO KẾT QUẢ ---")
print(f"Chỉ số R2 Score: {r2_final:.4f}")
print(f"Sai số trung bình (MAE): {mae:,.0f} ")

--- BÁO CÁO KẾT QUẢ ---
Chỉ số R2 Score: 0.8732
Sai số trung bình (MAE): 160,986 
